In [0]:
display(dbutils.fs.ls("s3://fraud-platform-batch-landing/"))

path,name,size,modificationTime
s3://fraud-platform-batch-landing/archive/,archive/,0,1783241132128
s3://fraud-platform-batch-landing/batch/,batch/,0,1783241132128


In [0]:
display(
    spark.read.option("header", True).csv(
        "s3://fraud-platform-batch-landing/batch/landing/ieee/train_transaction.csv"
    )
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
# ==========================
# Batch Configuration
# ==========================

SOURCE_PATH = "s3://fraud-platform-batch-landing/batch/landing/ieee"

BRONZE_DB = "fraud_bronze"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")

DataFrame[]

In [0]:
transaction_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(f"{SOURCE_PATH}/train_transaction.csv")
)

identity_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(f"{SOURCE_PATH}/train_identity.csv")
)

In [0]:
print("Transactions :", transaction_df.count())
print("Identity     :", identity_df.count())

display(transaction_df.limit(5))
display(identity_df.limit(5))

In [0]:
batch_id = current_timestamp()

transaction_df = (
    transaction_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("batch_id", batch_id)
    .withColumn("source_file", lit("train_transaction.csv"))
)

identity_df = (
    identity_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("batch_id", batch_id)
    .withColumn("source_file", lit("train_identity.csv"))
)

In [0]:
(transaction_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE_DB}.bronze_transactions"))

(identity_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE_DB}.bronze_identity"))

In [0]:
display(
    spark.sql(f"""
    SELECT COUNT(*)
    FROM {BRONZE_DB}.bronze_transactions
    """)
)

display(
    spark.sql(f"""
    SELECT COUNT(*)
    FROM {BRONZE_DB}.bronze_identity
    """)
)

COUNT(*)
590540


COUNT(*)
144233


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Customer deduplication
customer_window = Window.partitionBy(
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6"
).orderBy(col("TransactionDT").desc())

# Transaction columns only
customer_txn = (
    transaction_df
    .withColumn(
        "customer_id",
        sha2(
            concat_ws(
                "_",
                "card1",
                "card2",
                "card3",
                "card4",
                "card5",
                "card6"
            ),
            256
        )
    )
    .withColumn("rn", row_number().over(customer_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Identity lookup (ONLY required columns)
identity_lookup = identity_df.select(
    "TransactionID",
    "DeviceType",
    "DeviceInfo",
    "id_12",
    "id_13",
    "id_15",
    "id_16",
    "id_23",
    "id_27",
    "id_28",
    "id_29"
)

customer_master = (
    customer_txn
    .join(identity_lookup, on="TransactionID", how="left")
)

customer_master.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{BRONZE_DB}.bronze_customer_master")

print("Customer Master Created Successfully")
print(customer_master.count())

display(customer_master.limit(10))

Customer Master Created Successfully
14893


TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,ingestion_timestamp,batch_id,source_file,customer_id,DeviceType,DeviceInfo,id_12,id_13,id_15,id_16,id_23,id_27,id_28,id_29
3210739,0,5270458,27.0,W,1001,555.0,150.0,visa,226.0,debit,269.0,87.0,null,null,yahoo.com,null,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,4.0,2.0,51.0,51.0,20.0,360.0,20.0,null,null,null,null,360.0,null,null,null,null,360.0,null,null,null,M1,T,T,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,29.0,0.0,0.0,29.0,0.0,0.0,0.0,29.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-07-05T08:49:37.655Z,2026-07-05T08:49:37.655Z,train_transaction.csv,1b91b0449b34175735acce4d999b4ccfd479b2ce877476cc50ea4b62056f4020,null,null,null,null,null,null,null,null,null,null
3442358,0,11646701,226.0,W,1026,55

In [0]:
from pyspark.sql.functions import *

merchant_master = (
    transaction_df
    .select(
        "ProductCD",
        "card4",
        "card6",
        "addr2"
    )
    .dropDuplicates()
    .withColumn(
        "merchant_id",
        sha2(
            concat_ws(
                "_",
                "ProductCD",
                "card4",
                "card6",
                "addr2"
            ),
            256
        )
    )
)

In [0]:
merchant_master.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{BRONZE_DB}.bronze_merchant_master")

In [0]:
display(merchant_master.limit(10))

print("Merchant Count :", merchant_master.count())

ProductCD,card4,card6,addr2,merchant_id
W,visa,debit,87.0,4f2f0f42901343260e6397805b76724cdd5a0ba7b82306d1c154daa6bcd6b568
C,mastercard,debit,null,45c0c1f24542c2d1ecf44cd25ffcfdcc03877f06485a2e853c107734c92a80ab
H,visa,credit,87.0,fb928959833497911f4184a1a78dc8bcc68d2170b5c494e78ac393af4b676a37
W,discover,credit,87.0,88d6ce4c59cedcd91e999e533c367c947054e8eae4a657a159ca836cb6c3c8ef
W,mastercard,debit,87.0,34d3610cc7d1c2209a8cbf53f8733994f4b790f70fce91b502ad0f43bfded855
S,visa,debit,87.0,18f40ecea2bf97d0b03fed296977575aca9a5f3e647dc16840fd51904f455f00
R,visa,credit,87.0,d5311910b56193cacad49f1bfd36576bf6d6df0b03d3ccd0f8ded799ae827181
H,american express,debit,87.0,e873cdd019eabac35df71ef895a9736a9cbbdb711b60feace5b1252debf9c0db
W,discover,debit,87.0,515add40fbafc0771055770177bbb8db5fb59b8162fc9c0c5f3fcecef5be62d9
R,mastercard,debit,87.0,79a664082a3c65a9dcbfddd064a39be0760977efe649106df3c3a7577cae15f4


Merchant Count : 269


**Country Risk**

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

country_data = [

# ---------- LOW RISK ----------

(87,"United States","North America",20,"LOW",0,0),
(124,"Canada","North America",20,"LOW",0,0),
(826,"United Kingdom","Europe",20,"LOW",0,0),
(276,"Germany","Europe",20,"LOW",0,0),
(250,"France","Europe",20,"LOW",0,0),
(380,"Italy","Europe",20,"LOW",0,0),
(724,"Spain","Europe",20,"LOW",0,0),
(528,"Netherlands","Europe",20,"LOW",0,0),
(756,"Switzerland","Europe",20,"LOW",0,0),
(752,"Sweden","Europe",20,"LOW",0,0),
(578,"Norway","Europe",20,"LOW",0,0),
(208,"Denmark","Europe",20,"LOW",0,0),
(246,"Finland","Europe",20,"LOW",0,0),
(36,"Australia","Oceania",20,"LOW",0,0),
(554,"New Zealand","Oceania",20,"LOW",0,0),
(392,"Japan","Asia",20,"LOW",0,0),
(702,"Singapore","Asia",20,"LOW",0,0),
(410,"South Korea","Asia",20,"LOW",0,0),
(356,"India","Asia",20,"LOW",0,0),
(784,"United Arab Emirates","Middle East",20,"LOW",0,0),
(682,"Saudi Arabia","Middle East",20,"LOW",0,0),
(458,"Malaysia","Asia",20,"LOW",0,0),
(764,"Thailand","Asia",20,"LOW",0,0),
(372,"Ireland","Europe",20,"LOW",0,0),
(56,"Belgium","Europe",20,"LOW",0,0),
(40,"Austria","Europe",20,"LOW",0,0),
(620,"Portugal","Europe",20,"LOW",0,0),

# ---------- MEDIUM ----------

(76,"Brazil","South America",60,"MEDIUM",0,0),
(484,"Mexico","North America",60,"MEDIUM",0,0),
(360,"Indonesia","Asia",60,"MEDIUM",0,0),
(608,"Philippines","Asia",60,"MEDIUM",0,0),
(792,"Turkey","Europe",60,"MEDIUM",0,0),
(704,"Vietnam","Asia",60,"MEDIUM",0,0),
(710,"South Africa","Africa",60,"MEDIUM",0,0),

# ---------- HIGH ----------

(566,"Nigeria","Africa",90,"HIGH",0,0),
(586,"Pakistan","Asia",90,"HIGH",0,0),
(643,"Russia","Europe",90,"HIGH",1,1),
(408,"North Korea","Asia",95,"HIGH",1,1),
(364,"Iran","Middle East",95,"HIGH",1,1),
(760,"Syria","Middle East",95,"HIGH",1,1)

]

country_risk = spark.createDataFrame(
    country_data,
    [
        "country_code",
        "country_name",
        "region",
        "risk_score",
        "risk_level",
        "sanctions_flag",
        "ofac_watchlist"
    ]
).withColumn(
    "created_ts",
    current_timestamp()
)

spark.sql("DROP TABLE IF EXISTS fraud_feature_store.country_features")

country_risk.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("fraud_feature_store.country_features")

display(country_risk)

country_code,country_name,region,risk_score,risk_level,sanctions_flag,ofac_watchlist,created_ts
87,United States,North America,20,LOW,0,0,2026-07-05T12:21:28.471Z
124,Canada,North America,20,LOW,0,0,2026-07-05T12:21:28.471Z
826,United Kingdom,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z
276,Germany,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z
250,France,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z
380,Italy,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z
724,Spain,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z
528,Netherlands,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z
756,Switzerland,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z
752,Sweden,Europe,20,LOW,0,0,2026-07-05T12:21:28.471Z


**Exchange Rates**

In [0]:
from pyspark.sql import Row

exchange_rates = spark.createDataFrame([

    Row(currency_code="USD", currency_name="US Dollar", exchange_rate=1.0000),
    Row(currency_code="EUR", currency_name="Euro", exchange_rate=1.1700),
    Row(currency_code="GBP", currency_name="British Pound", exchange_rate=1.3600),
    Row(currency_code="INR", currency_name="Indian Rupee", exchange_rate=0.0120),
    Row(currency_code="JPY", currency_name="Japanese Yen", exchange_rate=0.0068),
    Row(currency_code="CAD", currency_name="Canadian Dollar", exchange_rate=0.7300),
    Row(currency_code="AUD", currency_name="Australian Dollar", exchange_rate=0.6600),
    Row(currency_code="CHF", currency_name="Swiss Franc", exchange_rate=1.2200),
    Row(currency_code="SGD", currency_name="Singapore Dollar", exchange_rate=0.7400),
    Row(currency_code="CNY", currency_name="Chinese Yuan", exchange_rate=0.1400),
    Row(currency_code="AED", currency_name="UAE Dirham", exchange_rate=0.2700),
    Row(currency_code="SAR", currency_name="Saudi Riyal", exchange_rate=0.2700),
    Row(currency_code="BRL", currency_name="Brazilian Real", exchange_rate=0.1800),
    Row(currency_code="ZAR", currency_name="South African Rand", exchange_rate=0.0550),
    Row(currency_code="NGN", currency_name="Nigerian Naira", exchange_rate=0.00065)

])

In [0]:
exchange_rates.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{BRONZE_DB}.bronze_exchange_rates")

display(exchange_rates)

currency_code,currency_name,exchange_rate
USD,US Dollar,1.0
EUR,Euro,1.17
GBP,British Pound,1.36
INR,Indian Rupee,0.012
JPY,Japanese Yen,0.0068
CAD,Canadian Dollar,0.73
AUD,Australian Dollar,0.66
CHF,Swiss Franc,1.22
SGD,Singapore Dollar,0.74
CNY,Chinese Yuan,0.14


**OFAC**

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

ofac = spark.createDataFrame([

    Row(watchlist_id=1, entity_name="John Smith", country="US", entity_type="Individual", risk_score=95, watchlist_source="OFAC"),
    Row(watchlist_id=2, entity_name="Ali Hassan", country="IR", entity_type="Individual", risk_score=100, watchlist_source="OFAC"),
    Row(watchlist_id=3, entity_name="Ivan Petrov", country="RU", entity_type="Individual", risk_score=90, watchlist_source="OFAC"),
    Row(watchlist_id=4, entity_name="Kim Jong Trading", country="KP", entity_type="Organization", risk_score=100, watchlist_source="OFAC"),
    Row(watchlist_id=5, entity_name="Global Export Ltd", country="SY", entity_type="Organization", risk_score=95, watchlist_source="OFAC"),
    Row(watchlist_id=6, entity_name="Mohammed Rahim", country="AF", entity_type="Individual", risk_score=88, watchlist_source="OFAC"),
    Row(watchlist_id=7, entity_name="Eastern Energy Corp", country="RU", entity_type="Organization", risk_score=92, watchlist_source="OFAC"),
    Row(watchlist_id=8, entity_name="Black Sea Logistics", country="RU", entity_type="Organization", risk_score=91, watchlist_source="OFAC"),
    Row(watchlist_id=9, entity_name="Desert Trading LLC", country="IR", entity_type="Organization", risk_score=94, watchlist_source="OFAC"),
    Row(watchlist_id=10, entity_name="Ahmed Khan", country="PK", entity_type="Individual", risk_score=85, watchlist_source="OFAC"),
    Row(watchlist_id=11, entity_name="Golden Bridge Ltd", country="CN", entity_type="Organization", risk_score=75, watchlist_source="OFAC"),
    Row(watchlist_id=12, entity_name="Victor Romanov", country="RU", entity_type="Individual", risk_score=89, watchlist_source="OFAC"),
    Row(watchlist_id=13, entity_name="Ocean Shipping Group", country="KP", entity_type="Organization", risk_score=98, watchlist_source="OFAC"),
    Row(watchlist_id=14, entity_name="Far East Holdings", country="CN", entity_type="Organization", risk_score=80, watchlist_source="OFAC"),
    Row(watchlist_id=15, entity_name="National Arms Co", country="IR", entity_type="Organization", risk_score=99, watchlist_source="OFAC")

]).withColumn("last_updated", current_timestamp())

In [0]:
ofac.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{BRONZE_DB}.bronze_ofac")

display(ofac)

watchlist_id,entity_name,country,entity_type,risk_score,watchlist_source,last_updated
1,John Smith,US,Individual,95,OFAC,2026-07-05T09:02:49.564Z
2,Ali Hassan,IR,Individual,100,OFAC,2026-07-05T09:02:49.564Z
3,Ivan Petrov,RU,Individual,90,OFAC,2026-07-05T09:02:49.564Z
4,Kim Jong Trading,KP,Organization,100,OFAC,2026-07-05T09:02:49.564Z
5,Global Export Ltd,SY,Organization,95,OFAC,2026-07-05T09:02:49.564Z
6,Mohammed Rahim,AF,Individual,88,OFAC,2026-07-05T09:02:49.564Z
7,Eastern Energy Corp,RU,Organization,92,OFAC,2026-07-05T09:02:49.564Z
8,Black Sea Logistics,RU,Organization,91,OFAC,2026-07-05T09:02:49.564Z
9,Desert Trading LLC,IR,Organization,94,OFAC,2026-07-05T09:02:49.564Z
10,Ahmed Khan,PK,Individual,85,OFAC,2026-07-05T09:02:49.564Z


In [0]:
from pyspark.sql import Row

pipeline_config = spark.createDataFrame([

    Row(
        pipeline_name="Fraud Batch Pipeline",
        batch_frequency="Daily",
        expected_rows=600000,
        max_failure_percent=5,
        alert_email="fraudops@bank.com",
        active="Y"
    ),

    Row(
        pipeline_name="Customer Master Refresh",
        batch_frequency="Daily",
        expected_rows=500000,
        max_failure_percent=2,
        alert_email="masterdata@bank.com",
        active="Y"
    ),

    Row(
        pipeline_name="Merchant Master Refresh",
        batch_frequency="Weekly",
        expected_rows=5000,
        max_failure_percent=2,
        alert_email="merchant@bank.com",
        active="Y"
    ),

    Row(
        pipeline_name="Country Risk Refresh",
        batch_frequency="Monthly",
        expected_rows=30,
        max_failure_percent=0,
        alert_email="risk@bank.com",
        active="Y"
    ),

    Row(
        pipeline_name="Exchange Rate Refresh",
        batch_frequency="Daily",
        expected_rows=15,
        max_failure_percent=0,
        alert_email="treasury@bank.com",
        active="Y"
    ),

    Row(
        pipeline_name="OFAC Refresh",
        batch_frequency="Daily",
        expected_rows=15,
        max_failure_percent=0,
        alert_email="compliance@bank.com",
        active="Y"
    )

])

In [0]:
pipeline_config.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{BRONZE_DB}.bronze_pipeline_config")

display(pipeline_config)

pipeline_name,batch_frequency,expected_rows,max_failure_percent,alert_email,active
Fraud Batch Pipeline,Daily,600000,5,fraudops@bank.com,Y
Customer Master Refresh,Daily,500000,2,masterdata@bank.com,Y
Merchant Master Refresh,Weekly,5000,2,merchant@bank.com,Y
Country Risk Refresh,Monthly,30,0,risk@bank.com,Y
Exchange Rate Refresh,Daily,15,0,treasury@bank.com,Y
OFAC Refresh,Daily,15,0,compliance@bank.com,Y
